# Import Libraries

In [13]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import os
from collections import OrderedDict, defaultdict
from datetime import datetime
import re
import math
from typing import Dict, List, Tuple, Any, Optional
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, average_precision_score, top_k_accuracy_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# Prepare Datasets

In [14]:
class BugDataset(Dataset):
    def __init__(self, similarities, labels, pairs):
        self.similarities = similarities
        self.labels = labels
        self.pairs = pairs
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        report_idx, code_idx = self.pairs[idx]
        features = self.similarities[report_idx, code_idx]
        label = self.labels[report_idx, code_idx]
        return features, label

In [15]:
ROOT_PATH = Path.cwd().parent / "E:/Desktop/ISE/Deep Learning/Challenge Task NLP 1 - Bug Localization/PreProcessing Data"

def load_dataset(project_name, test_size = 0.2):
    data = np.load(ROOT_PATH / f"prepared_data_{project_name}.npz", allow_pickle=True)
    similarities = torch.from_numpy(data['similarities']).float()
    Y = torch.from_numpy(data['Y']).float()

    num_reports = similarities.shape[0]
    num_codes = similarities.shape[1]

    report_indices = np.arange(num_reports)
    train_reports, test_reports = train_test_split(report_indices, test_size=test_size, random_state=42)

    train_pairs = np.array([(i, j) for i in train_reports for j in range(num_codes)])
    test_pairs = np.array([(i, j) for i in test_reports for j in range(num_codes)])

    train_dataset = BugDataset(similarities, Y, train_pairs)
    test_dataset = BugDataset(similarities, Y, test_pairs)
    return train_dataset, test_dataset, len(test_reports), num_codes

In [16]:
def make_dataloaders(train_dataset, test_dataset, num_codes, batch_size_train = 128):
    trainloader = DataLoader(train_dataset, batch_size=batch_size_train, shuffle=True)
    testloader = DataLoader(test_dataset, batch_size=num_codes, shuffle=False)
    return trainloader, testloader

# Imbalanced Bug Location Model

In [17]:
class net(nn.Module):
    def __init__(self, input_size = 3):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 300)
        self.fc2 = nn.Linear(300, 150)
        self.fc3 = nn.Linear(150, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        x = F.relu(self.fc2(x))
        x = self.dropout(x)

        x = torch.sigmoid(self.fc3(x))
        return x

# Calculate Focal Loss

In [18]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        eps = 1e-8
        inputs = torch.clamp(inputs, eps, 1 - eps)
        bce_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        pt = torch.where(targets == 1, inputs, 1 - inputs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

def create_model_and_loss():
    model = net(input_size=3)
    criterion = FocalLoss(alpha=0.75, gamma=2.0)
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
    
    return model, criterion, optimizer

# Train, Test and Evaluate

In [19]:
def calculate_bug_localization_metrics(model, testloader, device="cpu"):
    model.eval()
    
    bug_results = defaultdict(list) 
    
    with torch.no_grad():
        for features, labels in testloader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            
            batch_size = features.shape[0]
            
            for i in range(batch_size):
                report_idx = i // testloader.dataset.similarities.shape[1]  
                
                score = outputs[i].item()
                label = labels[i].item()
                
                bug_results[report_idx].append((score, label))
    
    top_k_accuracies = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 10: 0, 15: 0}
    mrr_scores = []
    map_scores = []
    
    num_reports = len(bug_results)
    
    for report_idx, results in bug_results.items():
        results.sort(key=lambda x: x[0], reverse=True)
        
        ranked_labels = [label for score, label in results]
        
        for k in [1, 2, 3, 4, 5, 10, 15]:
            if k <= len(ranked_labels):
                if sum(ranked_labels[:k]) > 0:
                    top_k_accuracies[k] += 1
        
        first_relevant_rank = None
        for rank, label in enumerate(ranked_labels, 1):
            if label == 1:
                first_relevant_rank = rank
                break
        
        if first_relevant_rank:
            mrr_scores.append(1.0 / first_relevant_rank)
        else:
            mrr_scores.append(0.0)
        
        relevant_files = [i for i, label in enumerate(ranked_labels) if label == 1]
        
        if len(relevant_files) > 0:
            precision_at_k = []
            for k in range(1, len(ranked_labels) + 1):
                num_relevant_in_top_k = sum(ranked_labels[:k])
                precision_k = num_relevant_in_top_k / k
                precision_at_k.append(precision_k)
            
            avg_precision = sum(precision_at_k[pos-1] for pos in relevant_files) / len(relevant_files)
            map_scores.append(avg_precision)
        else:
            map_scores.append(0.0)
    
    final_metrics = {}
    
    for k in [1, 2, 3, 4, 5, 10, 15]:
        final_metrics[f'Top-{k}'] = (top_k_accuracies[k] / num_reports) * 100
    
    final_metrics['MRR'] = sum(mrr_scores) / len(mrr_scores)
    
    final_metrics['MAP'] = sum(map_scores) / len(map_scores)
    
    return final_metrics

In [20]:
def calculate_single_report_metrics(ranked_labels):
    metrics = {}
    
    for k in [1, 2, 3, 4, 5, 10, 15]:
        if k <= len(ranked_labels):
            metrics[f'top_{k}'] = sum(ranked_labels[:k]) > 0
        else:
            metrics[f'top_{k}'] = sum(ranked_labels) > 0
    
    first_relevant_rank = None
    for rank, label in enumerate(ranked_labels, 1):
        if label == 1:
            first_relevant_rank = rank
            break
    
    metrics['mrr'] = 1.0 / first_relevant_rank if first_relevant_rank else 0.0
    
    relevant_positions = [i+1 for i, label in enumerate(ranked_labels) if label == 1]
    
    if len(relevant_positions) > 0:
        precision_sum = 0.0
        for pos in relevant_positions:
            precision_at_pos = sum(ranked_labels[:pos]) / pos
            precision_sum += precision_at_pos
        
        metrics['map'] = precision_sum / len(relevant_positions)
    else:
        metrics['map'] = 0.0
    
    return metrics

In [21]:
def evaluate_by_report(model, testloader, device="cpu"):
    model.eval()
    all_metrics = []
    
    report_data = defaultdict(list)
    
    with torch.no_grad():
        batch_idx = 0
        for features, labels in testloader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            
            scores = outputs.squeeze().cpu().numpy()
            true_labels = labels.squeeze().cpu().numpy()
            
            sorted_indices = np.argsort(scores)[::-1]
            ranked_labels = true_labels[sorted_indices]
            
            report_metrics = calculate_single_report_metrics(ranked_labels)
            all_metrics.append(report_metrics)
            
            batch_idx += 1
    
    aggregated_metrics = {}
    
    for k in [1, 2, 3, 4, 5, 10, 15]:
        top_k_hits = sum(1 for m in all_metrics if m[f'top_{k}'])
        aggregated_metrics[f'Top-{k}'] = (top_k_hits / len(all_metrics)) * 100
    
    aggregated_metrics['MRR'] = np.mean([m['mrr'] for m in all_metrics])
    aggregated_metrics['MAP'] = np.mean([m['map'] for m in all_metrics])
    
    return aggregated_metrics

In [ ]:
def train_model(model, criterion, optimizer, trainloader, testloader, save_path, device="cpu"):
    epochs = 10
    warm_epoch = 5
    init_lr = 1e-2
    last_lr = 1e-4
    T_max = epochs
    T_cur = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        epoch_steps = 0
        T_cur += 1
        
        if epoch <= warm_epoch:
            optimizer.param_groups[0]['lr'] = (1.0 * epoch) / warm_epoch * init_lr
        else: 
            optimizer.param_groups[0]['lr'] = last_lr + (init_lr - last_lr) * (1 + np.cos(T_cur * np.pi / T_max)) / 2

        for i, data in enumerate(trainloader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            outputs = outputs.squeeze()

            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            epoch_steps += 1
            
        avg_loss = running_loss / epoch_steps
        print(f"[Epoch {epoch:2d}] Loss: {avg_loss:.6f}")
        
        if epoch % 10 == 0:
            metrics = evaluate_by_report(model, testloader, device)
            print(f"[Epoch {epoch:2d}] Top-1: {metrics['Top-1']:.2f}%, "
                  f"Top-5: {metrics['Top-5']:.2f}%, MRR: {metrics['MRR']:.4f}")
                
    print("Finished Training")
    
    final_metrics = evaluate_by_report(model, testloader, device)
    print("\nFinal Test Results:")
    print(f"Top-1 Accuracy: {final_metrics['Top-1']:.2f}%")
    print(f"Top-2 Accuracy: {final_metrics['Top-2']:.2f}%")
    print(f"Top-3 Accuracy: {final_metrics['Top-3']:.2f}%")
    print(f"Top-4 Accuracy: {final_metrics['Top-4']:.2f}%")
    print(f"Top-5 Accuracy: {final_metrics['Top-5']:.2f}%")
    print(f"Top-10 Accuracy: {final_metrics['Top-10']:.2f}%")
    print(f"Top-15 Accuracy: {final_metrics['Top-15']:.2f}%")
    print(f"MRR: {final_metrics['MRR']:.4f}")
    print(f"MAP: {final_metrics['MAP']:.4f}")
    
    torch.save(model.state_dict(), save_path)
    return final_metrics

# Results

In [23]:
all_results = {}

def save_and_display_results():
    data = []
    projects = list(all_results.keys())
    
    for project in projects:
        if 'ImbalancedBugLoc' in all_results[project]:
            metrics = all_results[project]['ImbalancedBugLoc']
            row = {
                'Project': project.capitalize(),
                'Model': 'ImbalancedBugLoc',
                '1': f"{metrics['Top-1']:.1f}",
                '2': f"{metrics['Top-2']:.1f}",
                '3': f"{metrics['Top-3']:.1f}",
                '4': f"{metrics['Top-4']:.1f}",
                '5': f"{metrics['Top-5']:.1f}",
                '10': f"{metrics['Top-10']:.1f}",
                '15': f"{metrics['Top-15']:.1f}",
                'MRR': f"{metrics['MRR']:.2f}",
                'MAP': f"{metrics['MAP']:.2f}"
            }
            data.append(row)
    
    if data:
        df = pd.DataFrame(data)
        print("=== PERFORMANCE COMPARISON TABLE ===")
        print(df.to_string(index=False))
        
        df.to_csv('bug_localization_results.csv', index=False)
        print(f"\nResults saved to: {Path.cwd() / 'bug_localization_results.csv'}")
    
    return df if data else None

save_and_display_results()

In [24]:
import traceback
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

projects = ['aspectj', 'tomcat', 'eclipse', 'swt', 'birt']

for project in projects:
    print(f"\n=== Project: {project} ===")
    try:
        train_dataset, test_dataset, num_test_reports, num_codes = load_dataset(project)
        trainloader, testloader = make_dataloaders(train_dataset, test_dataset, num_codes, batch_size_train=128)
        print(f"Train pairs: {len(train_dataset)}, Test pairs: {len(test_dataset)}, Test reports: {num_test_reports}, Files per report: {num_codes}")

        model, criterion, optimizer = create_model_and_loss()
        model.to(device)

        save_path = Path.cwd() / f"model_{project}.pth"
        print(f"Saving model to: {save_path}")

        final_metrics = train_model(model, criterion, optimizer, trainloader, testloader, str(save_path), device)

        if project not in all_results:
            all_results[project] = {}
        all_results[project]['ImbalancedBugLoc'] = final_metrics

        print(f"Finished project {project}.")

    except Exception as e:
        print(f"Error for project {project}: {e}")
        traceback.print_exc()

print("\n" + "="*50)
print("FINAL RESULTS TABLE")
print("="*50)
save_and_display_results()

Device: cuda

=== Project: aspectj ===
Train pairs: 3275340, Test pairs: 822290, Test reports: 119, Files per report: 6910
Saving model to: e:\Desktop\ISE\Deep Learning\Challenge Task NLP 1 - Bug Localization\PreProcessing Data\model_aspectj.pth
[Epoch  1] Loss: 0.000587
[Epoch  2] Loss: 0.000326
[Epoch  3] Loss: 0.000283
[Epoch  4] Loss: 0.000240
[Epoch  5] Loss: 0.000214
[Epoch  6] Loss: 0.000203
[Epoch  7] Loss: 0.000199
[Epoch  8] Loss: 0.000197
[Epoch  9] Loss: 0.000198
[Epoch 10] Loss: 0.000197
[Epoch 10] Top-1: 55.46%, Top-5: 70.59%, MRR: 0.6199
Finished Training

Final Test Results:
Top-1 Accuracy: 55.46%
Top-2 Accuracy: 61.34%
Top-3 Accuracy: 63.87%
Top-4 Accuracy: 68.91%
Top-5 Accuracy: 70.59%
Top-10 Accuracy: 74.79%
Top-15 Accuracy: 76.47%
MRR: 0.6199
MAP: 0.6199
Finished project aspectj.

=== Project: tomcat ===
Train pairs: 1514136, Test pairs: 380328, Test reports: 212, Files per report: 1794
Saving model to: e:\Desktop\ISE\Deep Learning\Challenge Task NLP 1 - Bug Localiz

,Project,Model,1,2,3,4,5,10,15,MRR,MAP
0,Aspectj,ImbalancedBugLoc,55.5,61.3,63.9,68.9,70.6,74.8,76.5,0.62,0.62
1,Tomcat,ImbalancedBugLoc,65.6,77.4,81.6,84.4,86.3,90.1,91.5,0.75,0.75
2,Eclipse,ImbalancedBugLoc,38.9,45.0,48.6,51.6,52.7,57.2,59.2,0.45,0.45
3,Swt,ImbalancedBugLoc,25.6,31.4,34.7,38.5,41.0,50.3,55.8,0.33,0.33
4,Birt,ImbalancedBugLoc,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
